# Entraînement Kaggle — une tâche par session, de zéro

GPU Kaggle uniquement. Ne pas lancer ce notebook en local.

Choisir `TASK` : `vehicles`, `traffic_lights` ou `plates_detection`. Une session n'entraîne qu'une tâche.

`SMOKE = True` : 200 images, 3 époques. Le notebook affiche le temps de chaque époque et l'estimation pour 250 époques. `SMOKE = False` : entraînement complet.

`RESUME = True` copie `last.pt` depuis `RESUME_FROM` (un dataset Kaggle ou un dossier de `/kaggle/working`) puis reprend avec `resume=True`. `save_period=10` écrit un point de reprise toutes les 10 époques.

Avant `train`, le notebook échoue si un `.pt` inattendu est présent ou si le journal signale un téléchargement. Le seul fichier autorisé est le `last.pt` de reprise. `check_amp` est remplacé pour ne pas télécharger `yolo11n.pt`.

Les images COCO et les nôtres sont listées dans `sources.csv`. Après l'entraînement, précision, rappel et mAP50 sont calculés à part pour chaque source, par classe. Si `test_scenes/images` contient des images, elles sont évaluées.


In [ ]:
import torch
print(torch.__version__, "cuda", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("Pas de GPU : ne pas entraîner ici.")
print(torch.cuda.get_device_name(0))
%pip install -q ultralytics

In [ ]:
from pathlib import Path
import csv
import random
import shutil

import yaml

TASK = "vehicles"  # vehicles | traffic_lights | plates_detection
SMOKE = True
RESUME = False
RESUME_FROM = Path("/kaggle/input/previous-run")
SEED = 42
SMOKE_IMAGES = 200
FULL_EPOCHS = 250

OURS = Path("/kaggle/input/infraction-processed")
COCO = Path("/kaggle/working/coco_out")
if not COCO.is_dir():
    mounted = sorted(Path("/kaggle/input").glob("*/coco_out"))
    COCO = mounted[0] if mounted else COCO
MERGED = Path("/kaggle/working/merged")
RUNS = Path("/kaggle/working/runs")
EXPORTS = Path("/kaggle/working/exports")
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
NAMES = {
    "vehicles": ["car", "bus", "truck", "motorcycle"],
    "traffic_lights": ["traffic_light"],
    "plates_detection": ["plate"],
}
if TASK not in NAMES:
    raise SystemExit(f"TASK inconnu : {TASK}")
if not OURS.is_dir():
    raise SystemExit(f"Dataset local introuvable : {OURS}")
coco_ready = COCO.is_dir()
if not coco_ready and not SMOKE:
    raise SystemExit(f"Sortie COCO introuvable : {COCO}. Lancer d'abord kaggle_01_coco.")
if not coco_ready:
    print("SMOKE sans COCO : seules nos images sont utilisées.")

task_root = MERGED / TASK
if task_root.exists():
    shutil.rmtree(task_root)

def pairs(root):
    images = root / "images"
    labels = root / "labels"
    found = []
    if not images.is_dir():
        return found
    for image in sorted(images.iterdir()):
        if image.suffix.lower() not in IMAGE_EXTS:
            continue
        label = labels / f"{image.stem}.txt"
        found.append((image, label if label.is_file() else None))
    return found

def copy_pair(image, label, dest_root, split, prefix, source, journal):
    image_dir = dest_root / split / "images"
    label_dir = dest_root / split / "labels"
    image_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    name = f"{prefix}{image.name}"
    shutil.copy2(image, image_dir / name)
    text = label.read_text(encoding="utf-8") if label is not None and label.is_file() else ""
    (label_dir / f"{Path(name).stem}.txt").write_text(text, encoding="utf-8")
    journal.append({"image": name, "split": split, "source": source})

def proportions(root):
    counts = {}
    for split in ("train", "val", "test"):
        folder = root / split / "images"
        counts[split] = sum(1 for path in folder.iterdir() if path.suffix.lower() in IMAGE_EXTS) if folder.is_dir() else 0
    total = sum(counts.values()) or 1
    return {split: counts[split] / total for split in counts}

def assign_groups(items, fractions):
    names = list(fractions)
    order = list(items)
    random.Random(SEED).shuffle(order)
    targets = {name: fractions[name] * len(order) for name in names}
    counts = {name: 0 for name in names}
    assigned = {name: [] for name in names}
    for item in order:
        def score(name, _item=item):
            after = counts[name] + 1
            overflow = max(0.0, after - targets[name])
            return (overflow, abs(targets[name] - after))
        choice = min(names, key=score)
        assigned[choice].append(item)
        counts[choice] += 1
    return assigned

def write_yaml(root, names):
    payload = {
        "path": str(root),
        "train": "train/images",
        "val": "val/images",
        "test": "test/images",
        "names": {index: name for index, name in enumerate(names)},
    }
    (root / "data.yaml").write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding="utf-8")

journal = []
if TASK == "vehicles":
    for split in ("train", "val", "test"):
        for image, label in pairs(OURS / "vehicles" / split):
            copy_pair(image, label, task_root, split, "", "ours", journal)
    if coco_ready:
        placed = assign_groups(pairs(COCO / "vehicles_coco"), {"train": 0.70, "val": 0.20, "test": 0.10})
        for split, rows in placed.items():
            for image, label in rows:
                copy_pair(image, label, task_root, split, "coco_", "coco", journal)
elif TASK == "traffic_lights":
    light_fraction = proportions(OURS / "traffic_lights")
    print("proportions feux", {name: round(value, 4) for name, value in light_fraction.items()})
    for split in ("train", "val", "test"):
        for image, label in pairs(OURS / "traffic_lights" / split):
            copy_pair(image, label, task_root, split, "", "ours", journal)
    if coco_ready:
        for folder in ("coco_feux_positifs", "coco_negatifs_feux"):
            placed = assign_groups(pairs(COCO / folder), light_fraction)
            prefix = "pos_" if "positifs" in folder else "neg_"
            for split, rows in placed.items():
                for image, label in rows:
                    copy_pair(image, label, task_root, split, prefix, "coco", journal)
else:
    for split in ("train", "val", "test"):
        for image, label in pairs(OURS / "plates_detection" / split):
            copy_pair(image, label, task_root, split, "", "ours", journal)

if SMOKE:
    by_split = {split: [row for row in journal if row["split"] == split] for split in ("train", "val", "test")}
    total = len(journal) or 1
    quota = {split: int(round(SMOKE_IMAGES * len(by_split[split]) / total)) for split in by_split}
    while sum(quota.values()) > SMOKE_IMAGES:
        quota[max(quota, key=quota.get)] -= 1
    while sum(quota.values()) < SMOKE_IMAGES and sum(quota.values()) < len(journal):
        quota[max(by_split, key=lambda split: len(by_split[split]) - quota[split])] += 1
    rng = random.Random(SEED)
    kept = []
    for split, rows in by_split.items():
        order = list(rows)
        rng.shuffle(order)
        chosen = order[: quota[split]]
        kept.extend(chosen)
        for row in rows:
            if row not in chosen:
                (task_root / split / "images" / row["image"]).unlink(missing_ok=True)
                (task_root / split / "labels" / f"{Path(row['image']).stem}.txt").unlink(missing_ok=True)
    journal = kept
    print(f"SMOKE : {len(journal)} images")

write_yaml(task_root, NAMES[TASK])
with (task_root / "sources.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["image", "split", "source"])
    writer.writeheader()
    writer.writerows(journal)
counts = {split: sum(row["split"] == split for row in journal) for split in ("train", "val", "test")}
print(TASK, counts, "sources", {name: sum(row["source"] == name for row in journal) for name in ("ours", "coco")})


In [ ]:
import csv
import json
import logging
import shutil
from pathlib import Path

import yaml
import ultralytics.utils.checks as checks
import ultralytics.utils.downloads as downloads
import ultralytics.engine.trainer as trainer
from ultralytics.utils import WEIGHTS_DIR

def check_amp_without_weights(model):
    """AMP autorisé, sans télécharger yolo11n.pt ni yolov8n.pt."""
    return True

checks.check_amp = check_amp_without_weights
trainer.check_amp = check_amp_without_weights

download_log = []

class DownloadWatch(logging.Handler):
    def emit(self, record):
        text = record.getMessage()
        lowered = text.lower()
        if ".pt" in lowered and "download" in lowered:
            download_log.append(text)

watch = DownloadWatch()
logging.getLogger().addHandler(watch)
logging.getLogger("ultralytics").addHandler(watch)

def block_pt_download(file, *args, **kwargs):
    path = Path(str(file))
    if path.suffix.lower() == ".pt" and not path.is_file():
        raise AssertionError(f"téléchargement de poids bloqué : {file}")
    return original_attempt(file, *args, **kwargs)

def block_pt_url(url, *args, **kwargs):
    if ".pt" in str(url).lower():
        raise AssertionError(f"téléchargement de poids bloqué : {url}")
    return original_safe(url, *args, **kwargs)

original_attempt = downloads.attempt_download_asset
original_safe = downloads.safe_download
downloads.attempt_download_asset = block_pt_download
downloads.safe_download = block_pt_url

from ultralytics import YOLO

def list_pt(*roots):
    found = []
    for root in roots:
        folder = Path(root)
        if folder.is_file() and folder.suffix.lower() == ".pt":
            found.append(folder.resolve())
        elif folder.is_dir():
            found.extend(path.resolve() for path in folder.rglob("*.pt"))
    return found

def import_resume():
    if not RESUME:
        return None
    candidates = [
        RESUME_FROM / "weights" / "last.pt",
        RESUME_FROM / "last.pt",
        RESUME_FROM / TASK / "weights" / "last.pt",
        RESUME_FROM / TASK / "last.pt",
    ]
    source = next((path for path in candidates if path.is_file()), None)
    if source is None:
        raise SystemExit(f"last.pt introuvable dans {RESUME_FROM}")
    dest = RUNS / TASK / "weights" / "last.pt"
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, dest)
    for args_name in ("args.yaml",):
        args_source = source.parent.parent / args_name
        if args_source.is_file():
            shutil.copy2(args_source, dest.parent.parent / args_name)
    return dest.resolve()

allowed = import_resume()
scanned = list_pt(Path("/kaggle/working"), Path.cwd(), WEIGHTS_DIR)
unexpected = [str(path) for path in scanned if allowed is None or path != allowed]
report_dir = EXPORTS / TASK
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Vérification des poids",
    "",
    f"- Tâche : {TASK}",
    f"- Reprise : {RESUME}",
    f"- last.pt autorisé : {allowed or 'aucun'}",
    f"- Fichiers .pt trouvés avant l'entraînement : {len(scanned)}",
]
if unexpected:
    lines.append("- Échec : un .pt inattendu est présent.")
    lines.extend(f"  - {path}" for path in unexpected)
else:
    lines.append("- Aucun .pt inattendu dans le dossier de travail, le dossier courant, ni le dossier de poids Ultralytics.")
if download_log:
    lines.append("- Échec : le journal signale un téléchargement de .pt.")
    lines.extend(f"  - {text}" for text in download_log)
else:
    lines.append("- Le journal ne signale aucun téléchargement de .pt.")
(report_dir / "verification_poids.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))
if unexpected or download_log:
    raise AssertionError("poids inattendu ou téléchargé, voir verification_poids.md")

def architecture():
    for name in ("yolo11n.yaml", "yolov8n.yaml"):
        try:
            return YOLO(name)
        except Exception:
            continue
    raise SystemExit("Ni yolo11n.yaml ni yolov8n.yaml ne sont disponibles.")

if RESUME:
    model = YOLO(str(allowed))
else:
    model = architecture()
    if str(getattr(model, "ckpt_path", "") or "").endswith(".pt"):
        raise AssertionError(f"{TASK} : un poids .pt a été chargé.")
after_init = [str(path) for path in list_pt(Path("/kaggle/working"), Path.cwd(), WEIGHTS_DIR) if allowed is None or path != allowed]
if after_init or download_log:
    raise AssertionError(f"un .pt est apparu avant train : {after_init or download_log}")

epochs = 3 if SMOKE else FULL_EPOCHS
train_args = dict(
    data=str(task_root / "data.yaml"),
    epochs=epochs,
    patience=50,
    imgsz=640,
    seed=SEED,
    pretrained=False,
    project=str(RUNS),
    name=TASK,
    exist_ok=True,
    amp=True,
    save_period=10,
)
if RESUME:
    model.train(resume=True)
else:
    model.train(**train_args)

run_dir = RUNS / TASK
for relative in (
    "weights/best.pt",
    "weights/last.pt",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "results.csv",
):
    source = run_dir / relative
    if source.is_file():
        shutil.copy2(source, report_dir / source.name)
shutil.copy2(task_root / "sources.csv", report_dir / "sources.csv")

def read_results(csv_path):
    with csv_path.open(encoding="utf-8", newline="") as handle:
        return list(csv.DictReader(handle))

rows = read_results(run_dir / "results.csv") if (run_dir / "results.csv").is_file() else []
times = [float(row["time"]) for row in rows if row.get("time")]
deltas = []
if times:
    deltas = [times[0]] + [right - left for left, right in zip(times, times[1:])]
mean_epoch = sum(deltas) / len(deltas) if deltas else 0.0
estimate = mean_epoch * FULL_EPOCHS
print(f"temps par époque (s) : {[round(value, 1) for value in deltas]}")
print(f"moyenne {mean_epoch:.1f} s/époque, estimation {FULL_EPOCHS} époques : {estimate / 3600:.2f} h")

def class_metrics(result, names):
    per_class = {}
    box = result.box
    for index, name in enumerate(names):
        precision, recall, map50, map95 = box.class_result(index)
        per_class[name] = {
            "precision": float(precision),
            "recall": float(recall),
            "mAP50": float(map50),
            "mAP50-95": float(map95),
        }
    return per_class

def evaluate_folder(image_rows, folder_name, names):
    subset = task_root / folder_name
    if subset.exists():
        shutil.rmtree(subset)
    images = subset / "images"
    labels = subset / "labels"
    images.mkdir(parents=True)
    labels.mkdir(parents=True)
    for row in image_rows:
        origin = task_root / row["split"] / "images" / row["image"]
        label = task_root / row["split"] / "labels" / f"{Path(row['image']).stem}.txt"
        shutil.copy2(origin, images / row["image"])
        text = label.read_text(encoding="utf-8") if label.is_file() else ""
        (labels / f"{Path(row['image']).stem}.txt").write_text(text, encoding="utf-8")
    payload = {
        "path": str(subset),
        "train": "images",
        "val": "images",
        "names": {index: name for index, name in enumerate(names)},
    }
    data_yaml = subset / "data.yaml"
    data_yaml.write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding="utf-8")
    result = model.val(data=str(data_yaml), split="val", imgsz=640, plots=False, verbose=False)
    return class_metrics(result, names)

with (task_root / "sources.csv").open(encoding="utf-8", newline="") as handle:
    sources = list(csv.DictReader(handle))
by_source = {}
for source_name in ("ours", "coco"):
    rows_source = [row for row in sources if row["source"] == source_name and row["split"] == "val"]
    if not rows_source:
        print(source_name, "aucune image de validation")
        continue
    by_source[source_name] = evaluate_folder(rows_source, f"val_{source_name}", NAMES[TASK])
    print(source_name, by_source[source_name])

scene_candidates = [
    OURS / "test_scenes",
    Path("/kaggle/input/test-scenes"),
    Path("/kaggle/working/test_scenes"),
]
scene = next(
    (folder for folder in scene_candidates if (folder / "images").is_dir() and any((folder / "images").iterdir())),
    None,
)
scene_metrics = None
if scene is None:
    print("test_scenes vide : évaluation des scènes de Marrakech sautée.")
else:
    subset = task_root / "val_marrakech"
    if subset.exists():
        shutil.rmtree(subset)
    (subset / "images").mkdir(parents=True)
    (subset / "labels").mkdir(parents=True)
    kept = 0
    for image in sorted((scene / "images").iterdir()):
        if image.suffix.lower() not in IMAGE_EXTS:
            continue
        shutil.copy2(image, subset / "images" / image.name)
        label = scene / "labels" / f"{image.stem}.txt"
        text = label.read_text(encoding="utf-8") if label.is_file() else ""
        (subset / "labels" / f"{image.stem}.txt").write_text(text, encoding="utf-8")
        kept += 1
    payload = {
        "path": str(subset),
        "train": "images",
        "val": "images",
        "names": {index: name for index, name in enumerate(NAMES[TASK])},
    }
    (subset / "data.yaml").write_text(yaml.safe_dump(payload, allow_unicode=True, sort_keys=False), encoding="utf-8")
    scene_result = model.val(data=str(subset / "data.yaml"), split="val", imgsz=640, plots=False, verbose=False)
    scene_metrics = class_metrics(scene_result, NAMES[TASK])
    print("test_scenes", kept, scene_metrics)

last = rows[-1] if rows else {}
wanted = ["metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]
metrics = {key: float(last[key]) for key in wanted if key in last and last[key] != ""}
metrics["epochs_run"] = len(rows)
metrics["seconds_per_epoch"] = deltas
metrics["mean_seconds_per_epoch"] = mean_epoch
metrics["estimated_seconds_250_epochs"] = estimate
metrics["smoke"] = SMOKE
metrics["resume"] = RESUME
metrics["sources"] = by_source
metrics["test_scenes"] = scene_metrics
metrics["check_amp"] = "remplacé, aucun .pt téléchargé pour le test AMP"
(report_dir / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(TASK, {key: metrics[key] for key in metrics if key not in {"sources", "test_scenes"}})
